**Проект 1. Экосистемность подписки ПРАЙМ: Главный рост пользователей с большим количеством сервисов происходит не на этапе первого использования, а дальше — когда активные пользователи начинают пользоваться дополнительными сервисами**

Ценность экосистемной подписки определяется не только количеством клиентов, получивших доступ к ПРАЙМ, но и тем, насколько глубоко они вовлекаются в использование сервисов экосистемы.

Для ПРАЙМ важно не только добиться первого использования сервиса, но и переводить уже активировавшихся клиентов к использованию нескольких сервисов. Поэтому в работе рассматривается последовательная воронка вовлечения: от первого использования к 2+ и далее к 3+ сервисам.

Исследовательский вопрос: как меняется глубина использования экосистемы в первые 60 дней после получения доступа к ПРАЙМ и за счёт каких этапов воронки растёт число клиентов, использующих 3 и более сервисов?

Рабочая гипотеза: рост числа клиентов с 3+ сервисами может быть связан не только с первичной активацией — переходом от отсутствия использования к первому сервису, — но и с дальнейшим переходом уже активных пользователей от 1 к 2+ и от 2 к 3+ сервисам.

Дизайн исследования: рассматриваются клиенты, которые впервые получили доступ к ПРАЙМ в мае 2026 года как владельцы подписки (owner) или участники шеринга (member). Для каждого клиента отслеживается первый непрерывный период доступа к ПРАЙМ и активность в течение первых 60 дней после access_start.

Для сравнения динамики используются накопительные показатели за первые 30 и первые 60 дней. Основной анализ проводится на одной и той же выборке клиентов, сохранявших непрерывный доступ все 60 дней, чтобы изменение показателей не зависело от изменения состава выборки.

Поскольку тарифы предоставляют разное количество доступных сервисов, различия между тарифами рассматриваются отдельно. Факторная декомпозиция числа клиентов с 3+ сервисами также проводится в разрезе тарифов.

Период выгрузки событий: 1 мая — 31 июля 2026 года. Такой период позволяет полностью наблюдать первые 60 дней для клиентов, впервые получивших доступ в течение мая.

Зерно клиентской витрины: одна строка — один client_id.

Для каждого клиента рассчитываются:

- дата начала и окончания первого непрерывного периода доступа к ПРАЙМ;

- наличие полного 30- и 60-дневного периода наблюдения;

- роль первого доступа: owner или member;

- тариф и partner_id, связанные с первым доступом;

- время от получения доступа до первого использования сервиса;

- число usage-событий за первые 30 и 60 дней;

- число уникальных использованных сервисов за первые 30 и 60 дней;

- доля использованных сервисов от числа сервисов, доступных по тарифу;

- принадлежность к этапам воронки: использование 1+, 2+ и 3+ сервисов;

- дополнительные показатели транзакционной активности.

Ключевой результативный показатель — число клиентов, использующих 3 и более сервисов.

В основном анализе сравнивается одна и та же когорта клиентов с непрерывным доступом в течение всех 60 дней. Поэтому изменение числа клиентов с 3+ сервисами между 30-м и 60-м днём не связано с изменением размера выборки.

Сначала общий прирост числа клиентов с 3+ сервисами аддитивно разделяется по тарифам. Затем внутри каждого тарифа методом цепных подстановок оценивается вклад изменения трёх этапов воронки: первичной активации, перехода от 1 к 2+ сервисам и перехода от 2 к 3+ сервисам.


## 1. Проверка и подготовка данных

In [ ]:
from prime.config import get_engine
from sqlalchemy import text

engine = get_engine()

with engine.connect() as conn:
    result = conn.execute(text("select 1"))
    print(result.scalar())

In [ ]:
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()

def q(sql: str, **params) -> pd.DataFrame:
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()

START = "2026-05-01"
END = "2026-08-01"

print(
    "подключились как:",
    q("select current_user as login").iloc[0, 0]
)

Подключились к базе ПРАЙМ, зафиксировали единый часовой пояс UTC и задали окно выгрузки событий с 1 мая по 31 июля 2026 года.

In [ ]:
def check_dataframe(df):
    content = []
    n_nan = []

    for col in df.columns:
        s = df[col]
        n_nan.append(s.isna().sum())

        if pd.api.types.is_numeric_dtype(s):
            content.append(f"{s.min()} – {s.max()}")
        else:
            content.append(s.dropna().unique()[:10].tolist())

    return pd.DataFrame({
        "column": df.columns,
        "type": df.dtypes.values,
        "nunique": df.nunique(dropna=False).values,
        "content": content,
        "NaN": n_nan
    })

### 1.1. Загружаем таблицы, необходимые для определения первого доступа

In [ ]:
clients = q("""
    select *
    from prime.clients
""")

clients.head()

In [ ]:
subscriptions = q("""
    select *
    from prime.subscriptions
""")

In [ ]:
members = q("""
    select *
    from prime.subscription_members
""")

In [ ]:
subs_per_client = (
    subscriptions
    .groupby("owner_client_id")
    .agg(n_subscriptions=("subscription_id", "nunique"))
    .reset_index()
)

subs_per_client["n_subscriptions"].value_counts().sort_index()

Проверяем кратность `owner_client_id → subscription_id` до построения когорты. В дальнейшем `subscriptions` используется как источник характеристик собственной подписки владельца.

## 2. Формирование и очистка майской когорты

Клиент входит в когорту только если его **самый первый доступ к ПРАЙМ за всю доступную историю** приходится на май 2026 года.

Сначала независимо формируем кандидатов:
- `owner`: первая собственная подписка началась в мае;
- `member`: первое присоединение к любой подписке произошло в мае.

Затем делаем перекрёстную проверку:
- из майских owners исключаем тех, кто был member до мая;
- из майских members исключаем тех, кто был owner до мая.


In [ ]:
# Кандидаты-owner: первая собственная подписка началась в мае

first_owner_access = (
    subscriptions
    .sort_values("started_at")
    .drop_duplicates("owner_client_id", keep="first")
)

may_owner_candidates = first_owner_access[
    (first_owner_access["started_at"] >= "2026-05-01") &
    (first_owner_access["started_at"] < "2026-06-01")
].copy()

may_owner_candidates.shape

In [ ]:
# Кандидаты-member: самое первое joined_at по всей таблице members приходится на май

first_member_access = (
    members
    .sort_values("joined_at")
    .drop_duplicates("client_id", keep="first")
)

may_member_candidates = first_member_access[
    (first_member_access["joined_at"] >= "2026-05-01") &
    (first_member_access["joined_at"] < "2026-06-01")
].copy()

may_member_candidates.shape

In [ ]:
# Перекрёстная очистка истории ДО мая

prior_member_ids = members.loc[
    members["joined_at"] < "2026-05-01",
    "client_id"
].unique()

may_owner_candidates_clean = may_owner_candidates[
    ~may_owner_candidates["owner_client_id"].isin(prior_member_ids)
].copy()

prior_owner_ids = subscriptions.loc[
    subscriptions["started_at"] < "2026-05-01",
    "owner_client_id"
].unique()

may_member_candidates_clean = may_member_candidates[
    ~may_member_candidates["client_id"].isin(prior_owner_ids)
].copy()

print("Owner-кандидатов до проверки:", len(may_owner_candidates))
print("Owner-кандидатов после проверки:", len(may_owner_candidates_clean))
print("Исключено owners:", len(may_owner_candidates) - len(may_owner_candidates_clean))

print()

print("Member-кандидатов до проверки:", len(may_member_candidates))
print("Member-кандидатов после проверки:", len(may_member_candidates_clean))
print("Исключено members:", len(may_member_candidates) - len(may_member_candidates_clean))

После этой проверки май считается первым знакомством клиента с ПРАЙМ независимо от роли. Клиенты, которые уже имели доступ до мая через другой тип участия, в исследуемую когорту не попадают.

### 2.1. Приводим owners и members к единому зерну

In [ ]:
# Owners: сохраняем характеристики самого первого собственного доступа в мае

may_owners = may_owner_candidates_clean[
    [
        "owner_client_id",
        "subscription_id",
        "started_at",
        "tariff_id",
        "partner_id"
    ]
].copy()

may_owners = may_owners.rename(columns={
    "owner_client_id": "client_id",
    "started_at": "access_start"
})

may_owners["role"] = "owner"

may_owners = may_owners[
    [
        "client_id",
        "subscription_id",
        "role",
        "access_start",
        "tariff_id",
        "partner_id"
    ]
]

In [ ]:
# Members: добавляем характеристики подписки, через которую был получен первый доступ.
# Используем полную subscriptions, потому что сама подписка владельца могла начаться до мая.

subscription_info = subscriptions[
    [
        "subscription_id",
        "tariff_id",
        "partner_id"
    ]
].copy()

may_members = may_member_candidates_clean.merge(
    subscription_info,
    on="subscription_id",
    how="left",
    validate="m:1"
)

may_members["access_start"] = may_members["joined_at"]
may_members["role"] = "member"

may_members = may_members[
    [
        "client_id",
        "subscription_id",
        "role",
        "access_start",
        "tariff_id",
        "partner_id"
    ]
]

In [ ]:
# Собираем первые майские способы доступа и оставляем одну строку на клиента.
# Если клиент впервые стал и owner, и member внутри мая, берём более ранний access_start.

overlap_ids = set(may_owners["client_id"]) & set(may_members["client_id"])
print("Клиентов одновременно owner и member в мае:", len(overlap_ids))

first_access_rows = pd.concat(
    [may_owners, may_members],
    ignore_index=True
)

cohort_clients = (
    first_access_rows
    .sort_values(["client_id", "access_start"])
    .drop_duplicates("client_id", keep="first")
    .reset_index(drop=True)
)

cohort_clients["role"] = cohort_clients["role"].astype("category")

print("Строк:", len(cohort_clients))
print("Уникальных client_id:", cohort_clients["client_id"].nunique())
print("Дубликатов:", cohort_clients["client_id"].duplicated().sum())

### 2.2. Определяем непрерывный период доступа к ПРАЙМ

Для каждого клиента когорты собираем **все** известные интервалы доступа: собственную подписку (`owner`) и участия в чужих подписках (`member`).

Интервалы объединяются только если они пересекаются или соприкасаются. Поэтому `access_end` означает конец **первого непрерывного периода доступа**, а не просто самую позднюю дату среди всех подписок клиента.

Это позволяет не считать клиента активным в дни, когда между двумя подписками был реальный разрыв.

In [ ]:
# Все owner-интервалы доступа для клиентов нашей когорты

cohort_ids = cohort_clients["client_id"].unique()

owner_intervals = subscriptions[
    subscriptions["owner_client_id"].isin(cohort_ids)
][
    ["owner_client_id", "started_at", "ended_at"]
].copy()

owner_intervals = owner_intervals.rename(columns={
    "owner_client_id": "client_id",
    "started_at": "interval_start",
    "ended_at": "interval_end"
})

# Все member-интервалы доступа для этих же клиентов
member_intervals = members[
    members["client_id"].isin(cohort_ids)
].copy()

member_intervals = member_intervals.merge(
    subscriptions[["subscription_id", "ended_at"]],
    on="subscription_id",
    how="left",
    validate="m:1"
)

member_intervals["interval_start"] = member_intervals["joined_at"]

# Конец конкретного membership — более ранняя из дат:
# клиент вышел сам или закончилась сама подписка.
member_intervals["interval_end"] = (
    member_intervals[["left_at", "ended_at"]]
    .min(axis=1)
)

member_intervals = member_intervals[
    ["client_id", "interval_start", "interval_end"]
]

# Общая история доступов независимо от роли
access_intervals = pd.concat(
    [owner_intervals, member_intervals],
    ignore_index=True
).sort_values(["client_id", "interval_start"])

# Техническая проверка: конец интервала не должен быть раньше начала
invalid_intervals = access_intervals[
    access_intervals["interval_end"].notna() &
    (access_intervals["interval_end"] < access_intervals["interval_start"])
]

print("Некорректных интервалов доступа:", len(invalid_intervals))

In [ ]:
def get_first_continuous_access(group):
    """Возвращает первый непрерывный интервал доступа клиента к ПРАЙМ."""

    group = group.sort_values("interval_start").reset_index(drop=True)

    access_start = group.loc[0, "interval_start"]
    access_end = group.loc[0, "interval_end"]

    for _, row in group.iloc[1:].iterrows():
        next_start = row["interval_start"]
        next_end = row["interval_end"]

        # Если текущий интервал открыт, доступ уже считается непрерывным дальше.
        if pd.isna(access_end):
            break

        # Новый доступ начинается до окончания текущего или ровно в момент окончания:
        # разрыва нет, объединяем интервалы.
        if next_start <= access_end:
            if pd.isna(next_end):
                access_end = pd.NaT
                break

            access_end = max(access_end, next_end)

        # Если следующий доступ начинается позже — возник реальный разрыв.
        else:
            break

    return pd.Series({
        "continuous_access_start": access_start,
        "continuous_access_end": access_end
    })


continuous_access = (
    access_intervals
    .groupby("client_id")
    .apply(get_first_continuous_access, include_groups=False)
    .reset_index()
)

continuous_access.head()

In [ ]:
# Добавляем рассчитанный непрерывный интервал в финальную клиентскую витрину.
# access_start из первого способа доступа должен совпадать с continuous_access_start.

cohort_clients = cohort_clients.merge(
    continuous_access,
    on="client_id",
    how="left",
    validate="1:1"
)

start_mismatch = (
    cohort_clients["access_start"] != cohort_clients["continuous_access_start"]
).sum()

print("Несовпадений первой даты доступа:", start_mismatch)

# Используем рассчитанные даты как единые клиентские границы доступа.
cohort_clients["access_start"] = cohort_clients["continuous_access_start"]
cohort_clients["access_end"] = cohort_clients["continuous_access_end"]

cohort_clients = cohort_clients.drop(
    columns=["continuous_access_start", "continuous_access_end"]
)

# Был ли доступ непрерывным все первые 60 дней после первого подключения.
cohort_clients["full_60d_access"] = (
    cohort_clients["access_end"].isna()
    |
    (
        cohort_clients["access_end"]
        >= cohort_clients["access_start"] + pd.Timedelta(days=60)
    )
)

print("Строк:", len(cohort_clients))
print("Уникальных client_id:", cohort_clients["client_id"].nunique())
print("Дубликатов:", cohort_clients["client_id"].duplicated().sum())
print("С непрерывным доступом 60 дней:", cohort_clients["full_60d_access"].sum())

cohort_clients.head()


Зерно: **одна строка = один `client_id`**.

- `access_start` — первая дата доступа клиента к ПРАЙМ в мае 2026 года;
- `access_end` — конец первого **непрерывного** периода доступа; `NaT` означает, что этот период остаётся открытым;
- `full_60d_access` — был ли у клиента непрерывный доступ ко всем первым 60 дням наблюдения;
- `role`, `subscription_id`, `tariff_id` и `partner_id` относятся к самому раннему способу получения доступа.

Если после окончания первого непрерывного периода клиент позднее снова получил доступ к ПРАЙМ, этот поздний период не расширяет первый клиентский интервал через возникший разрыв.

## 3. Сервисы и использование

In [ ]:
services = q("""
    select *
    from prime.services
""")

services

При оценке глубины экосистемного использования необходимо учитывать тариф пользователя: набор доступных сервисов различается между тарифами. Поэтому напрямую сравнивать количество использованных сервисов у клиентов разных тарифов некорректно — часть сервисов для них изначально недоступна. Например, мобильная связь доступна только начиная со второго тарифа. В дальнейшем глубину использования следует либо анализировать отдельно по тарифам, либо нормировать на количество сервисов, доступных конкретному пользователю.

In [ ]:
usage = q("""
    select
        usage_id,
        client_id,
        service_id,
        started_at,
        ended_at,
        device_type
    from prime.service_usage
    where started_at >= :start
      and started_at < :end
""", start=START, end=END)

usage.head()

In [ ]:
client_ids = cohort_clients["client_id"].unique()

cohort_usage = usage[
    usage["client_id"].isin(client_ids)
].copy()

check_dataframe(cohort_usage)

### 3.1. Проверка временной логики usage

In [ ]:
usage_check = cohort_usage.merge(
    cohort_clients[
        ["client_id", "access_start", "access_end"]
    ],
    on="client_id",
    how="left",
    validate="m:1"
)

usage_check["valid_time"] = (
    (usage_check["started_at"] >= usage_check["access_start"]) &
    (
        usage_check["access_end"].isna() |
        (usage_check["started_at"] < usage_check["access_end"])
    )
)

invalid_usage = usage_check[
    ~usage_check["valid_time"]
].copy()

print(usage_check["valid_time"].value_counts(dropna=False))
print(f"Доля usage вне интервала доступа: {(~usage_check['valid_time']).mean() * 100:.3f}%")
print("Уникальных клиентов с такими usage:", invalid_usage["client_id"].nunique())

In [ ]:
# Для дальнейшего анализа используем только события внутри периода доступа.

cohort_usage = (
    usage_check.loc[usage_check["valid_time"], usage.columns]
    .copy()
)

cohort_usage["service_id"].value_counts()

После формирования когорты временная проверка выполняется до расчёта метрик. События вне периода доступа сохраняются отдельно в `invalid_usage` для аудита, а в основной анализ не включаются.

## 4. Транзакции

In [ ]:
transactions = q("""
    select
        transaction_id,
        client_id,
        occurred_at,
        amount,
        mcc_code,
        service_id,
        channel,
        status
    from prime.transactions
    where occurred_at >= :start
      and occurred_at < :end
""", start=START, end=END)

transactions.head()

In [ ]:
cohort_transactions = transactions[
    transactions["client_id"].isin(client_ids)
].copy()

# Для анализа покупательской активности оставляем успешные операции.
cohort_transactions = cohort_transactions[
    cohort_transactions["status"] == "success"
].copy()

check_dataframe(cohort_transactions)

### 4.1. Проверка временной логики транзакций

In [ ]:
transactions_check = cohort_transactions.merge(
    cohort_clients[
        ["client_id", "access_start", "access_end"]
    ],
    on="client_id",
    how="left",
    validate="m:1"
)

transactions_check["valid_time"] = (
    (transactions_check["occurred_at"] >= transactions_check["access_start"]) &
    (
        transactions_check["access_end"].isna() |
        (transactions_check["occurred_at"] < transactions_check["access_end"])
    )
)

invalid_transactions = transactions_check[
    ~transactions_check["valid_time"]
].copy()

print(transactions_check["valid_time"].value_counts(dropna=False))
print(
    f"Доля транзакций вне интервала доступа: "
    f"{(~transactions_check['valid_time']).mean() * 100:.3f}%"
)
print(
    "Уникальных клиентов с такими транзакциями:",
    invalid_transactions["client_id"].nunique()
)

In [ ]:
cohort_transactions = (
    transactions_check.loc[
        transactions_check["valid_time"],
        transactions.columns
    ]
    .copy()
)

### 4.2. Отрицательные суммы

In [ ]:
negative_transactions = cohort_transactions[
    cohort_transactions["amount"] < 0
].copy()

negative_share = (
    len(negative_transactions) / len(cohort_transactions) * 100
)

print("Всего транзакций:", len(cohort_transactions))
print("Отрицательных транзакций:", len(negative_transactions))
print(f"Доля отрицательных транзакций: {negative_share:.2f}%")
print(
    "Клиентов с отрицательными транзакциями:",
    negative_transactions["client_id"].nunique()
)

Отрицательные операции не рассматриваются как самостоятельные покупки. Для основной оценки покупательской активности они исключаются, а исходные наблюдения остаются в `negative_transactions` для контроля.

In [ ]:
cohort_transactions = cohort_transactions[
    cohort_transactions["amount"] >= 0
].copy()

### 4.3. Транзакции без service_id

In [ ]:
# NaN в service_id кодируем технической категорией 13.

cohort_transactions["service_id"] = (
    cohort_transactions["service_id"]
    .fillna(13)
    .astype("int64")
)

cohort_transactions["service_id"].value_counts(dropna=False)

In [ ]:
service_13_transactions = cohort_transactions[
    cohort_transactions["service_id"] == 13
].copy()

print("MCC-коды:")
display(service_13_transactions["mcc_code"].value_counts())

print("\nКаналы:")
display(service_13_transactions["channel"].value_counts())

Категория `service_id = 13` является технической так как по MCC-коду и каналу нельзя однозначно восстановить один из 12 сервисов.

Поэтому:
- `cohort_transactions` сохраняет все валидные транзакции клиента, включая техническую категорию 13;
- для анализа транзакций по конкретным сервисам используем только `service_id` 1–12.

In [ ]:
ecosystem_transactions = cohort_transactions[
    cohort_transactions["service_id"].between(1, 12)
].copy()

print("Все валидные транзакции:", len(cohort_transactions))
print("Транзакции, привязанные к сервисам 1–12:", len(ecosystem_transactions))

## 5. Подготовка данных о сервисах для витрины


In [ ]:
cohort_clients.head()

In [ ]:
cohort_clients.shape

In [ ]:
# Был ли доступ непрерывным все первые 30 дней после первого подключения
cohort_clients["full_30d_access"] = (
    cohort_clients["access_end"].isna()
    |
    (
        cohort_clients["access_end"]
        >= cohort_clients["access_start"] + pd.Timedelta(days=30)
    )
)


# Был ли доступ непрерывным все первые 60 дней после первого подключения
cohort_clients["full_60d_access"] = (
    cohort_clients["access_end"].isna()
    |
    (
        cohort_clients["access_end"]
        >= cohort_clients["access_start"] + pd.Timedelta(days=60)
    )
)

In [ ]:
print(
    "Full 30 days access:",
    cohort_clients["full_30d_access"].sum(),
    f"({cohort_clients['full_30d_access'].mean():.1%})"
)

print(
    "Full 60 days access:",
    cohort_clients["full_60d_access"].sum(),
    f"({cohort_clients['full_60d_access'].mean():.1%})"
)

В сформированной когорте 97,6% клиентов сохраняли непрерывный доступ к ПРАЙМ не менее 30 дней, а 93,4% — не менее 60 дней. Таким образом, основная часть пользователей остаётся в экосистеме на протяжении как минимум двух первых 30-дневных циклов, тогда как доля клиентов с доступом менее 30 дней составляет около 2,4%, а с доступом от 30 до 60 дней — около 4,2%.

В дальнейшем эти группы можно рассматривать отдельно и сравнивать их клиентские пути внутри экосистемы: последовательность подключения сервисов, переход от одного сервиса к нескольким и соответствующую воронку использования. Это позволит проверить, отличается ли глубина освоения экосистемы у клиентов, которые остаются менее 30 дней, от 30 до 60 дней и 60 дней и более.

**Подготовка таблиц с сервисами и транзакциями**

для **usage_client**:
- client_id

- days_to_first_usage

- n_usage_events_30d
- n_services_30d

- n_usage_events_60d
- n_services_60d

In [ ]:
# 1. Добавляем даты доступа клиента
usage_events = cohort_usage.merge(
    cohort_clients[
        [
            "client_id",
            "access_start",
            "access_end",
            "full_30d_access",
            "full_60d_access"
        ]
    ],
    on="client_id",
    how="left",
    validate="m:1"
)


# 2. Оставляем usage только внутри первых 60 дней доступа
# и только пока у клиента действительно был непрерывный доступ
usage_events = usage_events[
    (usage_events["started_at"] >= usage_events["access_start"])
    & (
        usage_events["started_at"]
        < usage_events["access_start"] + pd.Timedelta(days=60)
    )
    & (
        usage_events["access_end"].isna()
        | (usage_events["started_at"] < usage_events["access_end"])
    )
].copy()

# Находим первое использование каждого клиента
usage_client = (
    usage_events
    .groupby("client_id", as_index=False)
    .agg(
        first_usage_at=("started_at", "min")
    )
)

# Добавляем начало доступа
usage_client = usage_client.merge(
    cohort_clients[
        ["client_id", "access_start"]
    ],
    on="client_id",
    how="left",
    validate="1:1"
)

# Считаем количество дней до первого использования
usage_client["days_to_first_usage"] = (
    usage_client["first_usage_at"]
    - usage_client["access_start"]
).dt.total_seconds() / 86400

usage_client.head()

In [ ]:
import numpy as np
# 4. Считаем, сколько дней прошло от подключения до каждого usage-события
usage_events["days_from_access"] = (
    usage_events["started_at"]
    - usage_events["access_start"]
).dt.total_seconds() / 86400


# 5. Выделяем события первых 30 дней
# Берём только клиентов, у которых полностью наблюдаются первые 30 дней
usage_30d = usage_events[
    (usage_events["days_from_access"] < 30)
    & usage_events["full_30d_access"]
].copy()


# 6. Агрегируем показатели за первые 30 дней до уровня клиента
usage_30d = (
    usage_30d
    .groupby("client_id", as_index=False)
    .agg(
        n_usage_events_30d=("usage_id", "nunique"),
        n_services_30d=("service_id", "nunique")
    )
)


# 7. Выделяем события первых 60 дней
# Здесь первые 60 дней включают и события первых 30 дней
# Берём только клиентов, у которых полностью наблюдаются все 60 дней
usage_60d = usage_events[
    (usage_events["days_from_access"] < 60)
    & usage_events["full_60d_access"]
].copy()


# 8. Агрегируем показатели за первые 60 дней до уровня клиента
usage_60d = (
    usage_60d
    .groupby("client_id", as_index=False)
    .agg(
        n_usage_events_60d=("usage_id", "nunique"),
        n_services_60d=("service_id", "nunique")
    )
)


# 9. Добавляем показатели первых 30 дней в usage_client
usage_client = usage_client.merge(
    usage_30d,
    on="client_id",
    how="outer",
    validate="1:1"
)


# 10. Добавляем показатели первых 60 дней в usage_client
usage_client = usage_client.merge(
    usage_60d,
    on="client_id",
    how="outer",
    validate="1:1"
)


# 11. Смотрим результат
usage_client.head()


In [ ]:
usage_client = usage_client[
    [
        "client_id",
        "days_to_first_usage",
        "n_usage_events_30d",
        "n_services_30d",
        "n_usage_events_60d",
        "n_services_60d"
    ]
]

In [ ]:
usage_client_full = cohort_clients[
    [
        "client_id",
        "full_30d_access",
        "full_60d_access"
    ]
].merge(
    usage_client,
    on="client_id",
    how="left",
    validate="1:1"
)

# Если первые 30 дней полностью наблюдаемы,
# но usage не было — ставим 0
mask_30 = usage_client_full["full_30d_access"]

usage_client_full.loc[
    mask_30,
    ["n_usage_events_30d", "n_services_30d"]
] = (
    usage_client_full.loc[
        mask_30,
        ["n_usage_events_30d", "n_services_30d"]
    ]
    .fillna(0)
)


# Если все 60 дней полностью наблюдаемы,
# но usage не было — ставим 0
mask_60 = usage_client_full["full_60d_access"]

usage_client_full.loc[
    mask_60,
    ["n_usage_events_60d", "n_services_60d"]
] = (
    usage_client_full.loc[
        mask_60,
        ["n_usage_events_60d", "n_services_60d"]
    ]
    .fillna(0)
)

usage_client = usage_client_full[
    [
        "client_id",
        "days_to_first_usage",
        "n_usage_events_30d",
        "n_services_30d",
        "n_usage_events_60d",
        "n_services_60d"
    ]
]
usage_client.head()

In [ ]:
usage_client.shape

Таблица `usage_client` успешно приведена к зерну «одна строка — один клиент». Для каждого клиента рассчитано время до первого использования сервиса, а также накопленное число usage-событий и уникальных сервисов за первые 30 и 60 дней. Если клиент наблюдался весь соответствующий период, но не использовал сервисы, значения корректно равны 0; если первое использование не произошло в течение первых 60 дней, `days_to_first_usage` остаётся `NaN`.

для **transactions_client**:

- client_id

- days_to_first_transaction

- n_transactions_30d
- transaction_amount_30d
- avg_transaction_30d
- n_transaction_services_30d

- n_transactions_60d
- transaction_amount_60d
- avg_transaction_60d
- n_transaction_services_60d

In [ ]:
# 1. Добавляем к транзакциям информацию о доступе клиента
transactions_events = cohort_transactions.merge(
    cohort_clients[
        [
            "client_id",
            "access_start",
            "access_end",
            "full_30d_access",
            "full_60d_access"
        ]
    ],
    on="client_id",
    how="left",
    validate="m:1"
)


# 2. Оставляем только транзакции:
# - после подключения клиента к ПРАЙМ
# - в пределах первых 60 дней
# - внутри непрерывного периода доступа

transactions_events = transactions_events[
    (transactions_events["occurred_at"] >= transactions_events["access_start"])
    & (
        transactions_events["occurred_at"]
        < transactions_events["access_start"] + pd.Timedelta(days=60)
    )
    & (
        transactions_events["access_end"].isna()
        | (transactions_events["occurred_at"] < transactions_events["access_end"])
    )
].copy()


# 3. Находим первую транзакцию клиента в течение первых 60 дней
transactions_client = (
    transactions_events
    .groupby("client_id", as_index=False)
    .agg(
        first_transaction_at=("occurred_at", "min")
    )
)


# 4. Добавляем дату начала доступа,
# чтобы посчитать время до первой транзакции
transactions_client = transactions_client.merge(
    cohort_clients[
        ["client_id", "access_start"]
    ],
    on="client_id",
    how="left",
    validate="1:1"
)


# 5. Считаем количество дней от подключения
# до первой транзакции
transactions_client["days_to_first_transaction"] = (
    transactions_client["first_transaction_at"]
    - transactions_client["access_start"]
).dt.total_seconds() / 86400


# 6. Создаём техническую колонку для подсчёта сервисов
# Сервис 13 — это "сервис не определён",
# поэтому в число уникальных сервисов его не включаем
# При этом сама транзакция остаётся в количестве и сумме транзакций

transactions_events["service_id_for_count"] = (
    transactions_events["service_id"]
    .where(
        transactions_events["service_id"].between(1, 12)
    )
)


# 7. Выделяем транзакции первых 30 дней
# Только для клиентов, у которых полностью наблюдаются первые 30 дней

transactions_30d = transactions_events[
    (
        transactions_events["occurred_at"]
        < transactions_events["access_start"] + pd.Timedelta(days=30)
    )
    & transactions_events["full_30d_access"]
].copy()


# 8. Агрегируем транзакции за первые 30 дней
transactions_30d = (
    transactions_30d
    .groupby("client_id", as_index=False)
    .agg(
        n_transactions_30d=("transaction_id", "nunique"),
        transaction_amount_30d=("amount", "sum"),
        avg_transaction_30d=("amount", "mean"),
        n_transaction_services_30d=("service_id_for_count", "nunique")
    )
)


# 9. Выделяем транзакции первых 60 дней
# Это накопительный период: он включает первые 30 дней
# Берём только клиентов с полными 60 днями доступа

transactions_60d = transactions_events[
    transactions_events["full_60d_access"]
].copy()


# 10. Агрегируем транзакции за первые 60 дней
transactions_60d = (
    transactions_60d
    .groupby("client_id", as_index=False)
    .agg(
        n_transactions_60d=("transaction_id", "nunique"),
        transaction_amount_60d=("amount", "sum"),
        avg_transaction_60d=("amount", "mean"),
        n_transaction_services_60d=("service_id_for_count", "nunique")
    )
)


# 11. Пока оставляем в transactions_client
# только client_id и время до первой транзакции

transactions_client = transactions_client[
    [
        "client_id",
        "days_to_first_transaction"
    ]
]


# 12. Добавляем показатели за первые 30 дней
transactions_client = transactions_client.merge(
    transactions_30d,
    on="client_id",
    how="outer",
    validate="1:1"
)


# 13. Добавляем показатели за первые 60 дней
transactions_client = transactions_client.merge(
    transactions_60d,
    on="client_id",
    how="outer",
    validate="1:1"
)


# 14. Теперь создаём каркас на основе всей нашей когорты,
# чтобы в transactions_client была строка для каждого клиента

transactions_client_full = cohort_clients[
    [
        "client_id",
        "full_30d_access",
        "full_60d_access"
    ]
].merge(
    transactions_client,
    on="client_id",
    how="left",
    validate="1:1"
)


# 15. Если первые 30 дней полностью наблюдались,
# но транзакций не было:
# количество транзакций = 0
# сумма = 0
# количество сервисов = 0
#
# Средний чек оставляем NaN,
# потому что без транзакций средний чек не определён

mask_30 = transactions_client_full["full_30d_access"]

transactions_client_full.loc[
    mask_30,
    [
        "n_transactions_30d",
        "transaction_amount_30d",
        "n_transaction_services_30d"
    ]
] = (
    transactions_client_full.loc[
        mask_30,
        [
            "n_transactions_30d",
            "transaction_amount_30d",
            "n_transaction_services_30d"
        ]
    ]
    .fillna(0)
)


# 16. Аналогично для первых 60 дней

mask_60 = transactions_client_full["full_60d_access"]

transactions_client_full.loc[
    mask_60,
    [
        "n_transactions_60d",
        "transaction_amount_60d",
        "n_transaction_services_60d"
    ]
] = (
    transactions_client_full.loc[
        mask_60,
        [
            "n_transactions_60d",
            "transaction_amount_60d",
            "n_transaction_services_60d"
        ]
    ]
    .fillna(0)
)


# 17. Формируем финальную transactions_client
# full_30d_access и full_60d_access уже находятся в cohort_clients,
# поэтому здесь повторно их не храним

transactions_client = transactions_client_full[
    [
        "client_id",
        "days_to_first_transaction",

        "n_transactions_30d",
        "transaction_amount_30d",
        "avg_transaction_30d",
        "n_transaction_services_30d",

        "n_transactions_60d",
        "transaction_amount_60d",
        "avg_transaction_60d",
        "n_transaction_services_60d"
    ]
].copy()


# 18. Проверяем зерно таблицы:
# одна строка должна соответствовать одному клиенту

print("Строк в transactions_client:", len(transactions_client))

print(
    "Уникальных client_id:",
    transactions_client["client_id"].nunique()
)

print(
    "Одна строка на клиента:",
    transactions_client["client_id"].is_unique
)

transactions_client.head()

Таблица `transactions_client` приведена к зерну «одна строка — один клиент»: 7 033 строки соответствуют 7 033 уникальным `client_id`. Для каждого клиента рассчитано время до первой транзакции, а также накопленные показатели транзакционной активности за первые 30 и 60 дней: количество операций, сумма, средний чек и число сервисов с транзакциями. Если наблюдаемый период был полным, но транзакций не было, количественные показатели принимают значение 0, а средний чек остаётся `NaN`. Неопределённый `service_id = 13` не учитывается при подсчёте числа сервисов.

## 6. Сборка витрины

In [ ]:
mart = cohort_clients.copy()

print("ПРОТОКОЛ СБОРКИ ВИТРИНЫ")
print("Зерно: одна строка = один client_id")
print("Строк в каркасе:", len(mart))
print("Уникальных клиентов:", mart["client_id"].nunique())

In [ ]:
assert len(mart) == mart["client_id"].nunique()

print("1. Каркас cohort_clients — OK")

In [ ]:
rows_before = len(mart)

mart = mart.merge(
    usage_client,
    on="client_id",
    how="left",
    validate="1:1"
)

print("\n+ usage_client")
print("Строк до:", rows_before)
print("Строк после:", len(mart))
print(
    "Клиентов без usage-данных:",
    mart["days_to_first_usage"].isna().sum()
)

assert len(mart) == rows_before
assert mart["client_id"].is_unique

print("Зерно сохранено — OK")


In [ ]:
rows_before = len(mart)

mart = mart.merge(
    transactions_client,
    on="client_id",
    how="left",
    validate="1:1"
)

print("\n+ transactions_client")
print("Строк до:", rows_before)
print("Строк после:", len(mart))
print(
    "Клиентов без транзакций за первые 60 дней:",
    mart["days_to_first_transaction"].isna().sum()
)

assert len(mart) == rows_before
assert mart["client_id"].is_unique

print("Зерно сохранено — OK")

In [ ]:
print("\nИТОГОВАЯ ПРОВЕРКА")

print("Строк в витрине:", len(mart))
print(
    "Уникальных client_id:",
    mart["client_id"].nunique()
)
print(
    "Одна строка на клиента:",
    mart["client_id"].is_unique
)


Итоговая клиентская витрина успешно собрана с сохранением исходного зерна: 7 033 строки соответствуют 7 033 уникальным клиентам. После присоединения `usage_client` и `transactions_client` количество строк не изменилось, что подтверждает отсутствие размножения записей при merge. В когорте 2 205 клиентов не имели usage-событий в рассматриваемом окне, а 3 022 клиента не совершали транзакций в первые 60 дней. Таким образом, витрина корректно сохраняет как активных клиентов, так и клиентов без фактического использования сервисов или транзакционной активности.

## 7. Подсчет метрик на основе витрины 

In [ ]:
# Среднее и медиана количества использованных сервисов
# за первые 30 и 60 дней

services_stats = pd.DataFrame({
    "period": ["30d", "60d"],

    "n_clients": [
        mart["full_30d_access"].sum(),
        mart["full_60d_access"].sum()
    ],

    "mean_n_services": [
        mart.loc[
            mart["full_30d_access"],
            "n_services_30d"
        ].mean(),

        mart.loc[
            mart["full_60d_access"],
            "n_services_60d"
        ].mean()
    ],

    "median_n_services": [
        mart.loc[
            mart["full_30d_access"],
            "n_services_30d"
        ].median(),

        mart.loc[
            mart["full_60d_access"],
            "n_services_60d"
        ].median()
    ]
})

services_stats

анализ чувствительности 

In [ ]:
# Тест на чувствительность:
# сравниваем 30 и 60 дней на одной и той же выборке клиентов,
# которые имеют полные 60 дней наблюдения

same_cohort = mart[
    mart["full_60d_access"]
].copy()


services_stats_same_cohort = pd.DataFrame({
    "period": ["30d", "60d"],

    # В обоих случаях количество клиентов одинаковое
    "n_clients": [
        same_cohort["client_id"].nunique(),
        same_cohort["client_id"].nunique()
    ],

    "mean_n_services": [
        same_cohort["n_services_30d"].mean(),
        same_cohort["n_services_60d"].mean()
    ],

    "median_n_services": [
        same_cohort["n_services_30d"].median(),
        same_cohort["n_services_60d"].median()
    ]
})

services_stats_same_cohort

In [ ]:
# Берём только клиентов, у которых полностью наблюдаются 60 дней,
# чтобы 30 и 60 дней сравнивались на одинаковой выборке

same_cohort = mart[
    mart["full_60d_access"]
].copy()


# Статистика за 30 дней
stats_30d = (
    same_cohort
    .groupby("tariff_id", as_index=False)
    .agg(
        n_clients=("client_id", "nunique"),
        mean_services=("n_services_30d", "mean"),
        median_services=("n_services_30d", "median")
    )
)

stats_30d["period"] = "30d"


# Статистика за 60 дней
stats_60d = (
    same_cohort
    .groupby("tariff_id", as_index=False)
    .agg(
        n_clients=("client_id", "nunique"),
        mean_services=("n_services_60d", "mean"),
        median_services=("n_services_60d", "median")
    )
)

stats_60d["period"] = "60d"


# Объединяем
services_by_tariff_period = pd.concat(
    [stats_30d, stats_60d],
    ignore_index=True
)


# Для удобства сортируем
services_by_tariff_period = (
    services_by_tariff_period
    .sort_values(["tariff_id", "period"])
    .reset_index(drop=True)
)


services_by_tariff_period

На сопоставимой выборке из 6 569 клиентов среднее число используемых сервисов выросло с 0,93 за первые 30 дней до 1,66 за первые 60 дней, тогда как медиана осталась равной 1. Это подтверждает, что рост среднего не связан с изменением состава выборки и происходит за счёт части клиентов, которые со временем начинают использовать дополнительные сервисы.

В разрезе тарифов динамика различается: для тарифа 1 среднее число сервисов выросло с 0,87 до 1,51 на выборке из 3 766 клиентов, а для тарифа 2 — с 1,01 до 1,87 на выборке из 2 803 клиентов. Клиенты тарифа 2 используют больше сервисов и демонстрируют более сильный прирост, однако напрямую интерпретировать это как более высокую вовлечённость пока нельзя, поскольку тарифы предоставляют доступ к разному количеству сервисов. Для корректного сравнения далее стоит учитывать долю использованных сервисов от числа доступных по тарифу.

In [ ]:
# Количество сервисов, доступных по каждому тарифу
available_services = {
    1: 5,
    2: 12
}

mart["n_available_services"] = (
    mart["tariff_id"].map(available_services)
)


# Доля доступных сервисов, использованных за первые 30 дней
mart["service_share_30d"] = (
    mart["n_services_30d"]
    / mart["n_available_services"]
)


# Доля доступных сервисов, использованных за первые 60 дней
mart["service_share_60d"] = (
    mart["n_services_60d"]
    / mart["n_available_services"]
)

In [ ]:
# Одинаковая выборка для сравнения 30 и 60 дней

same_cohort = mart[
    mart["full_60d_access"]
].copy()
# Доля использованных сервисов за первые 30 дней

share_30d = (
    same_cohort
    .groupby("tariff_id", as_index=False)
    .agg(
        n_clients=("client_id", "nunique"),
        mean_service_share=("service_share_30d", "mean"),
        median_service_share=("service_share_30d", "median")
    )
)

share_30d["period"] = "30d"

# Доля использованных сервисов за первые 60 дней

share_60d = (
    same_cohort
    .groupby("tariff_id", as_index=False)
    .agg(
        n_clients=("client_id", "nunique"),
        mean_service_share=("service_share_60d", "mean"),
        median_service_share=("service_share_60d", "median")
    )
)

share_60d["period"] = "60d"

services_share_by_tariff_period = pd.concat(
    [share_30d, share_60d],
    ignore_index=True
)

services_share_by_tariff_period = (
    services_share_by_tariff_period
    .sort_values(["tariff_id", "period"])
    .reset_index(drop=True)
)

services_share_by_tariff_period


При учёте числа доступных по тарифу сервисов картина меняется: клиенты тарифа 1 осваивают большую долю доступной экосистемы, чем клиенты тарифа 2. На выборке из 3 766 клиентов тарифа 1 средняя доля использованных сервисов выросла с 17,3% за первые 30 дней до 30,2% за первые 60 дней. Для тарифа 2 на выборке из 2 803 клиентов показатель вырос с 8,4% до 15,6%.

При этом медиана в обоих тарифах за 30 и 60 дней не изменилась: 20% для тарифа 1 и 8,3% для тарифа 2. Это указывает на то, что рост среднего значения обеспечивается не всей клиентской базой, а частью пользователей, которые со временем начинают осваивать дополнительные сервисы. Таким образом, хотя клиенты тарифа 2 в абсолютном выражении используют больше сервисов, клиенты тарифа 1 используют большую долю доступных им возможностей подписки.

In [ ]:
import matplotlib.pyplot as plt

# Берём только клиентов с полными 60 днями доступа,
# чтобы сравнение 30 и 60 дней шло на одной и той же выборке
same_cohort = mart[
    mart["full_60d_access"]
].copy()


# -----------------------------
# ГРАФИК 1. Частота числа сервисов за первые 30 дней
# -----------------------------

# Считаем квартили
q25_30 = same_cohort["n_services_30d"].quantile(0.25)
q50_30 = same_cohort["n_services_30d"].quantile(0.50)
q75_30 = same_cohort["n_services_30d"].quantile(0.75)

# Частоты количества сервисов
freq_30 = (
    same_cohort["n_services_30d"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 6))
plt.bar(freq_30.index, freq_30.values)

# Линии квартилей
plt.axvline(q25_30, linestyle="--", label=f"Q25 = {q25_30:.2f}")
plt.axvline(q50_30, linestyle="--", label=f"Медиана = {q50_30:.2f}")
plt.axvline(q75_30, linestyle="--", label=f"Q75 = {q75_30:.2f}")

plt.title("Распределение числа использованных сервисов за первые 30 дней")
plt.xlabel("Количество использованных сервисов")
plt.ylabel("Частота клиентов")
plt.legend()
plt.show()


print("30 дней")
print("Размер выборки:", same_cohort["n_services_30d"].notna().sum())
print("Q25:", q25_30)
print("Медиана:", q50_30)
print("Q75:", q75_30)


# -----------------------------
# ГРАФИК 2. Частота числа сервисов за первые 60 дней
# -----------------------------

# Считаем квартили
q25_60 = same_cohort["n_services_60d"].quantile(0.25)
q50_60 = same_cohort["n_services_60d"].quantile(0.50)
q75_60 = same_cohort["n_services_60d"].quantile(0.75)

# Частоты количества сервисов
freq_60 = (
    same_cohort["n_services_60d"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 6))
plt.bar(freq_60.index, freq_60.values)

# Линии квартилей
plt.axvline(q25_60, linestyle="--", label=f"Q25 = {q25_60:.2f}")
plt.axvline(q50_60, linestyle="--", label=f"Медиана = {q50_60:.2f}")
plt.axvline(q75_60, linestyle="--", label=f"Q75 = {q75_60:.2f}")

plt.title("Распределение числа использованных сервисов за первые 60 дней")
plt.xlabel("Количество использованных сервисов")
plt.ylabel("Частота клиентов")
plt.legend()
plt.show()


print("60 дней")
print("Размер выборки:", same_cohort["n_services_60d"].notna().sum())
print("Q25:", q25_60)
print("Медиана:", q50_60)
print("Q75:", q75_60)

На сопоставимой выборке из 6 569 клиентов распределение числа используемых сервисов постепенно смещается вправо. За первые 30 дней медиана составляет 1 сервис, Q25 — 0, Q75 — 1; за первые 60 дней медиана остаётся равной 1, Q25 также остаётся на уровне 0, однако Q75 увеличивается до 2 сервисов.

Это означает, что типичное поведение большинства клиентов существенно не меняется: значительная часть пользователей по-прежнему либо не использует сервисы, либо ограничивается одним. При этом к 60-му дню увеличивается группа более вовлечённых клиентов, использующих несколько сервисов экосистемы. 

In [ ]:
# Берём клиентов, у которых первое использование произошло
# в течение первых 60 дней
first_usage_days = mart[
    "days_to_first_usage"
].dropna()


# Считаем квартили
q25_usage = first_usage_days.quantile(0.25)
q50_usage = first_usage_days.quantile(0.50)
q75_usage = first_usage_days.quantile(0.75)


# Строим распределение
plt.figure(figsize=(10, 6))

plt.hist(
    first_usage_days,
    bins=30
)

# Добавляем квартили
plt.axvline(
    q25_usage,
    linestyle="--",
    label=f"Q25 = {q25_usage:.1f} дней"
)

plt.axvline(
    q50_usage,
    linestyle="--",
    label=f"Медиана = {q50_usage:.1f} дней"
)

plt.axvline(
    q75_usage,
    linestyle="--",
    label=f"Q75 = {q75_usage:.1f} дней"
)


plt.title(
    "Распределение времени от подключения до первого использования сервиса"
)

plt.xlabel(
    "Дней от подключения до первого использования"
)

plt.ylabel(
    "Частота клиентов"
)

plt.legend()

plt.show()


# Выводим численные характеристики
print(
    "Клиентов с использованием в первые 60 дней:",
    first_usage_days.count()
)

print(
    "Клиентов без использования в первые 60 дней:",
    mart["days_to_first_usage"].isna().sum()
)

print("Q25:", round(q25_usage, 2))
print("Медиана:", round(q50_usage, 2))
print("Q75:", round(q75_usage, 2))

Из 7 033 клиентов 4 828 (68,6%) воспользовались хотя бы одним сервисом в течение первых 60 дней, тогда как 2 205 клиентов (31,4%) не начали пользоваться сервисами за рассматриваемый период.

Среди активировавшихся клиентов первое использование чаще происходит в начале клиентского пути: 25% начинают пользоваться сервисами в течение первых 6,3 дня, медианное время до первого использования составляет 15,2 дня, а 75% клиентов активируются в течение 30,8 дня. Распределение заметно смещено вправо: по мере увеличения времени с момента подключения число впервые активирующихся клиентов постепенно снижается.

Таким образом, первые недели после подключения являются ключевым периодом активации. При этом около четверти всех активировавшихся клиентов начинают пользоваться сервисами только после первого месяца, а почти треть всей когорты не использует ни одного сервиса даже за 60 дней.

In [ ]:
# Клиенты, которые впервые использовали сервис
# в течение первых 6 дней и имеют полные 60 дней наблюдения

early_activated = mart[
    mart["full_60d_access"]
    & (mart["days_to_first_usage"] <= 6)
].copy()


# Среднее и медиана числа сервисов за 30 и 60 дней
early_activated_stats = pd.DataFrame({
    "period": ["30d", "60d"],

    "n_clients": [
        early_activated["client_id"].nunique(),
        early_activated["client_id"].nunique()
    ],

    "mean_n_services": [
        early_activated["n_services_30d"].mean(),
        early_activated["n_services_60d"].mean()
    ],

    "median_n_services": [
        early_activated["n_services_30d"].median(),
        early_activated["n_services_60d"].median()
    ]
})


early_activated_stats

Среди 1 089 клиентов, которые впервые воспользовались сервисом в течение первых 6 дней и имели полные 60 дней наблюдения, глубина использования экосистемы заметно выше, чем в общей когорте. За первые 30 дней они используют в среднем 2,41 сервиса при медиане 2, а за первые 60 дней — уже 3,43 сервиса при медиане 3.

Для сравнения, по всей сопоставимой 60-дневной когорте среднее число сервисов составляло около 0,93 за 30 дней и 1,66 за 60 дней, при медиане 1 в обоих случаях. Это указывает на сильную связь между ранним первым использованием и последующей глубиной освоения экосистемы. При этом данный результат показывает ассоциацию, а не причинный эффект ранней активации.

In [ ]:
# Берём только клиентов:
# 1. с полными 60 днями наблюдения
# 2. которые хотя бы раз использовали сервис в первые 60 дней

activation_services = mart[
    mart["full_60d_access"]
    & mart["days_to_first_usage"].notna()
].copy()


# Переводим дробное количество дней в целый день первой активации
# Например:
# 6.3 дня -> 6-й день
# 15.8 дня -> 15-й день

activation_services["first_usage_day"] = (
    activation_services["days_to_first_usage"]
    .astype(int)
)


# Для каждого дня считаем:
# - количество клиентов
# - среднее число уникальных сервисов за 60 дней
# - медиану для дополнительного контроля

services_by_activation_day = (
    activation_services
    .groupby(
        "first_usage_day",
        as_index=False
    )
    .agg(
        n_clients=("client_id", "nunique"),
        mean_services_60d=("n_services_60d", "mean"),
        median_services_60d=("n_services_60d", "median")
    )
)


# Строим столбчатую диаграмму:
# X = день первого использования
# Y = среднее число сервисов за 60 дней

plt.figure(figsize=(14, 6))

plt.bar(
    services_by_activation_day["first_usage_day"],
    services_by_activation_day["mean_services_60d"]
)

plt.title(
    "Среднее число используемых сервисов снижается "
    "по мере более поздней первой активации"
)

plt.xlabel(
    "День первого использования после подключения"
)

plt.ylabel(
    "Среднее число уникальных сервисов за 60 дней"
)

plt.xticks(
    range(0, 60, 5)
)

plt.show()


Наблюдается выраженная обратная связь между временем до первого использования и глубиной освоения экосистемы: чем раньше клиент начинает пользоваться первым сервисом, тем больше уникальных сервисов он в среднем использует за первые 60 дней.

У клиентов, активировавшихся в первые дни после подключения, среднее число сервисов близко к 3–4, тогда как у клиентов с первым использованием ближе к 50–60-му дню показатель снижается примерно до 1 сервиса. При этом зависимость выглядит не строго линейной: наиболее сильное снижение происходит в первые недели, после чего кривая становится более пологой.

Это указывает на то, что ранняя активация связана с более глубоким вовлечением в экосистему. Однако по этому графику нельзя утверждать причинность: у рано активировавшихся клиентов также объективно больше времени для освоения дополнительных сервисов в пределах 60-дневного окна.

In [ ]:
# Берём одну и ту же выборку клиентов для 30 и 60 дней
funnel_base = mart[
    mart["full_60d_access"]
].copy()


# Считаем абсолютное количество клиентов на каждом шаге воронки
funnel = pd.DataFrame(
    {
        "clients": [
            len(funnel_base),
            len(funnel_base)
        ],

        "1plus_service": [
            (funnel_base["n_services_30d"] >= 1).sum(),
            (funnel_base["n_services_60d"] >= 1).sum()
        ],

        "2plus_services": [
            (funnel_base["n_services_30d"] >= 2).sum(),
            (funnel_base["n_services_60d"] >= 2).sum()
        ],

        "3plus_services": [
            (funnel_base["n_services_30d"] >= 3).sum(),
            (funnel_base["n_services_60d"] >= 3).sum()
        ]
    },
    index=["30d", "60d"]
)


# Смотрим абсолютные значения
funnel

In [ ]:
# Конверсия шага = доля от предыдущего шага
conv = funnel.div(
    funnel.shift(axis=1)
)

# Для первого столбца конверсия не нужна
conv.iloc[:, 1:].style.format(
    "{:.1%}"
).background_gradient(
    cmap="Greens",
    axis=None
)

На одной и той же 60-дневной когорте к 60-му дню увеличиваются как доля активировавшихся клиентов, так и доля пользователей 2+ и 3+ сервисов. Однако часть этого роста может быть связана с различиями во времени первой активации.

In [ ]:

# Подготовка факторов для факторного анализа
# Зерно итоговой таблицы:
# одна строка = тариф × период

# 1. Берём только клиентов,
# у которых полностью наблюдаются все первые 60 дней

factor_base = mart[
    mart["full_60d_access"]
].copy()


# 2. Создаём функцию, которая считает воронку
# для одного тарифа и одного периода

def calc_funnel(data, tariff_id, period):

    # Оставляем клиентов выбранного тарифа
    tariff_data = data[
        data["tariff_id"] == tariff_id
    ].copy()

    # Выбираем колонку количества сервисов
    # в зависимости от периода
    services_col = f"n_services_{period}"

    # Всего клиентов данного тарифа
    n_clients = tariff_data["client_id"].nunique()

    # Использовали хотя бы 1 сервис
    n_1plus = (
        tariff_data[services_col] >= 1
    ).sum()

    # Использовали 2+ сервиса
    n_2plus = (
        tariff_data[services_col] >= 2
    ).sum()

    # Использовали 3+ сервиса
    n_3plus = (
        tariff_data[services_col] >= 3
    ).sum()


    # Конверсия из всей базы в первое использование
    activation_rate = (
        n_1plus / n_clients
        if n_clients > 0
        else 0
    )

    # Конверсия из 1+ сервиса в 2+ сервиса
    conversion_1_to_2 = (
        n_2plus / n_1plus
        if n_1plus > 0
        else 0
    )

    # Конверсия из 2+ сервисов в 3+ сервиса
    conversion_2_to_3 = (
        n_3plus / n_2plus
        if n_2plus > 0
        else 0
    )


    return {
        "tariff_id": tariff_id,
        "period": period,

        "n_clients": n_clients,
        "n_1plus": n_1plus,
        "n_2plus": n_2plus,
        "n_3plus": n_3plus,

        "activation_rate": activation_rate,
        "conversion_1_to_2": conversion_1_to_2,
        "conversion_2_to_3": conversion_2_to_3
    }


# 3. Считаем показатели
# для двух тарифов и двух периодов

factor_rows = []

for tariff_id in [1, 2]:
    for period in ["30d", "60d"]:

        factor_rows.append(
            calc_funnel(
                factor_base,
                tariff_id,
                period
            )
        )


# 4. Собираем итоговую таблицу

factor_table = pd.DataFrame(
    factor_rows
)

factor_table

In [ ]:

# Проверяем, что сумма двух тарифов
# совпадает с общей воронкой

check_totals = (
    factor_table
    .groupby("period", as_index=False)
    .agg(
        n_clients=("n_clients", "sum"),
        n_1plus=("n_1plus", "sum"),
        n_2plus=("n_2plus", "sum"),
        n_3plus=("n_3plus", "sum")
    )
)

check_totals

## 8. Факторный анализ

Для каждого тарифа используется мультипликативная модель:

$$
N_{3+}
=
N \times Activation \times Conversion_{1\to2} \times Conversion_{2\to3}
$$

где:

- $N$ — число клиентов в рассматриваемой выборке;
- $Activation$ — доля клиентов, использовавших хотя бы один сервис;
- $Conversion_{1\to2}$ — доля клиентов с 2+ сервисами среди использовавших 1+ сервис;
- $Conversion_{2\to3}$ — доля клиентов с 3+ сервисами среди использовавших 2+ сервиса.

Влияние факторов рассчитывается методом цепных подстановок в порядке:

$$
N
\rightarrow
Activation
\rightarrow
Conversion_{1\to2}
\rightarrow
Conversion_{2\to3}
$$

In [ ]:
# 1. ГДЕ изменилось число клиентов с 3+ сервисами
# Аддитивное разложение по тарифам

where_by_tariff = (
    factor_table
    .pivot(
        index="tariff_id",
        columns="period",
        values="n_3plus"
    )
)

where_by_tariff["delta"] = (
    where_by_tariff["60d"]
    - where_by_tariff["30d"]
)

where_by_tariff

In [ ]:
# Общий рост должен быть равен сумме роста двух тарифов

total_delta = 1606 - 675

assert (
    abs(where_by_tariff["delta"].sum() - total_delta) < 0.01
), "Сумма изменений по тарифам не равна общему изменению"

print("Общее изменение:", total_delta)
print("Сумма по тарифам:", where_by_tariff["delta"].sum())

In [ ]:
from prime.factors import contrib

In [ ]:
# 2. ЗА СЧЁТ ЧЕГО изменилось число клиентов с 3+ сервисами
# Мультипликативное разложение внутри каждого тарифа

ORDER = [
    "clients",
    "activation",
    "conv_1_to_2",
    "conv_2_to_3"
]


effects_by_tariff = {}


for tariff_id in [1, 2]:

    # Значения на 30-й день
    row_30 = factor_table[
        (factor_table["tariff_id"] == tariff_id)
        & (factor_table["period"] == "30d")
    ].iloc[0]

    # Значения на 60-й день
    row_60 = factor_table[
        (factor_table["tariff_id"] == tariff_id)
        & (factor_table["period"] == "60d")
    ].iloc[0]


    # Факторы на 30-й день
    x0 = {
        "clients": row_30["n_clients"],
        "activation": row_30["activation_rate"],
        "conv_1_to_2": row_30["conversion_1_to_2"],
        "conv_2_to_3": row_30["conversion_2_to_3"]
    }


    # Факторы на 60-й день
    x1 = {
        "clients": row_60["n_clients"],
        "activation": row_60["activation_rate"],
        "conv_1_to_2": row_60["conversion_1_to_2"],
        "conv_2_to_3": row_60["conversion_2_to_3"]
    }


    effects_by_tariff[tariff_id] = contrib(
        x0,
        x1,
        order=ORDER
    )

In [ ]:
effects_table = pd.DataFrame(
    effects_by_tariff
).T

effects_table.index.name = "tariff_id"

effects_table["total_effect"] = (
    effects_table.sum(axis=1)
)

effects_table.round(1)

In [ ]:

# Доли факторов в общем приросте
factor_shares = pd.Series({
    "activation": effects_table["activation"].sum(),
    "conv_1_to_2": effects_table["conv_1_to_2"].sum(),
    "conv_2_to_3": effects_table["conv_2_to_3"].sum()
})

factor_shares = (
    factor_shares
    / total_delta
    * 100
)

factor_shares.round(1)

In [ ]:

# Доля каждого тарифа в общем приросте

tariff_shares = (
    effects_table["total_effect"]
    / total_delta
    * 100
)

tariff_shares.round(1)

In [ ]:
# Сумма всех факторных эффектов по двум тарифам
# должна дать общий прирост клиентов с 3+ сервисами

factor_total = effects_table["total_effect"].sum()

assert abs(
    factor_total - total_delta
) < 0.01, "Факторное разложение не сходится с общим изменением"

print(
    "Сумма влияний факторов:",
    round(factor_total, 1)
)

print(
    "Фактическое изменение:",
    total_delta
)

In [ ]:
effects_table

In [ ]:
from prime.factors import waterfall


# Суммируем влияние факторов по двум тарифам

factor_deltas = pd.Series({
    "Активация": effects_table["activation"].sum(),
    "1 → 2+ сервиса": effects_table["conv_1_to_2"].sum(),
    "2 → 3+ сервиса": effects_table["conv_2_to_3"].sum()
})


# Строим waterfall

ax = waterfall(
    factor_deltas,
    title="Рост числа клиентов с 3+ сервисами в основном связан с углублением использования"
)

# В функции по умолчанию стоит "тыс. ₽",
# поэтому меняем единицу измерения

ax.set_ylabel("Количество клиентов")

plt.show()

За первые 60 дней число клиентов, использующих 3 и более сервисов, выросло на 931 человека: с 675 до 1 606. Факторное разложение показывает, что 26,5% прироста связано с ростом первичной активации, 35,7% — с переходом пользователей от 1 к 2+ сервисам, а 37,8% — с переходом от 2 к 3+ сервисам. Таким образом, основная часть роста мультисервисной аудитории связана не с увеличением клиентской базы, а с углублением использования экосистемы уже активированными пользователями.

рост мультисервисной аудитории происходил на всех этапах воронки, но большая его часть связана не с первичной активацией, а с дальнейшим углублением использования


На активацию приходится 26.5% прироста, а на два последующих перехода вместе — около 73.5%. То есть основной резерв роста в рассматриваемый период находился уже после первого использования сервиса.

Если бы получилось наоборот, например:

| Фактор | Доля в приросте |
|---|---:|
| Активация | 20% |
|1 → 2+|55%|
|2 → 3+|25%|

это означало бы, что основной рост происходил в середине воронки: ПРАЙМ хорошо переводит клиента от первого сервиса ко второму, но дальнейший переход от двух сервисов к трём развивается слабее. Тогда 2 → 3+ можно было бы рассматривать как потенциальное узкое место.
